# 1. Data preparation

This notebook cleans and harmonises the five raw Excel files (2023, 2024, 2025, 2026 round 1, 2026 round 2) and checks the validity of the record linkage. The next two notebooks (Part A and Part B) use only the cleaned data produced here.

> **The data are not included in this repository.** The raw files contain candidate names and must not be published. Put the five Excel files in `data/raw/` (the expected file names and column layouts are in `src/exitexam/config.py`) before running.

**Principles (so that rounds are never mixed up):**
- Every round has one fixed key (`exam`): `2023`, `2024`, `2025`, `2026-R1`, `2026-R2`. No table is ever combined without this key.
- Columns are read by **position**, because the files have different layouts (table below).
- Every removed row gets exactly **one reason**, and the counts are recorded in the `audit` table.
- Every assumption is checked with an `assert` (the pass mark of 216, one row per person per round).
- Names are used only to link candidates. Output files contain no names or candidate numbers, only a hashed `person_id`.

In [1]:
import os, sys, warnings
sys.path.insert(0, os.path.abspath('../src'))          # the notebooks live in notebooks/, the code in src/
import numpy as np
import pandas as pd
from exitexam import config as C, cleaning as cl, linking as lk, stats_utils as su, plotting as P, pipeline
from exitexam import analysis_a as A, analysis_b as B

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 60); pd.set_option('display.width', 220)
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
P.setup()

## 1.1 Structure of the five files

| Round | Columns | Institution information | Other |
|---|---|---|---|
| 2023 | ID, name, father's name, grandfather's name, field, **year of graduation**, score, result, province | not available (province is always Kabul) | graduation year |
| 2024 | same, but **gender** instead of graduation year | not available (province is always Kabul) | gender (all male) |
| 2025 | ID, name, father's name, grandfather's name, programme, score, result | **available** (public / private / abroad, province, institute, field) | - |
| 2026 R1 | same | **available** | - |
| 2026 R2 | same | **available** (medicine only) | - |

So **Part A** covers the three rounds where the institute is known (2025, 2026 R1, 2026 R2). **Part B** covers all five rounds, where only field and score are shared.

In [2]:
clean, excluded, audit = cl.build_clean()
audit

exclusion_reason,rows_raw,score_missing,score_zero,result_annulled,result_unreadable,kept,ambiguous_names_kept
exam,,,,,,,
2023,6885,3,0,0,0,6882,0
2024,4381,0,0,0,0,4381,0
2025,9349,0,12,0,0,9337,0
2026-R1,6945,11,1,0,0,6933,0
2026-R2,6058,1,7,0,0,6050,0


## 1.2 Exclusion rules

Rows with an **empty score** (absent) or a **zero score** (cheating) are removed. The rules are applied in this order and each row is counted under its first reason only:

1. `score_missing`: the score is empty
2. `score_zero`: the score is zero
3. `result_annulled`: the result was annulled (cheating, biometric, quarantine) but the score is positive
4. `result_unreadable`: the result is not recognised

The table below shows what kind of result the removed rows had. The numbers are small and do not change the conclusions.

In [3]:
excluded.groupby(['exam', 'exclusion_reason', 'status']).size().rename('rows').reset_index()

,exam,exclusion_reason,status,rows
0,2023,score_missing,void,3
1,2025,score_zero,fail,4
2,2025,score_zero,void,8
3,2026-R1,score_missing,void,11
4,2026-R1,score_zero,fail,1
5,2026-R2,score_missing,unknown,1
6,2026-R2,score_zero,void,7


In [4]:
# checks that must hold, otherwise the analysis would be invalid
h = lk.add_history(clean)                                      # sorts strictly by round; asserts one row per person per round
assert (clean.groupby('exam').score.min() > 0).all()           # no zero / empty scores left
assert ((clean.passed) == (clean.score >= C.CUTOFF)).all()     # the 216 pass mark holds in every round
print('Pass mark checked in every round: pass <=> score >=', C.CUTOFF)
clean.groupby('exam').score.describe()

Pass mark checked in every round: pass <=> score >= 216.0


,count,mean,std,min,25%,50%,75%,max
exam,,,,,,,,
2023,"6,882.000",179.834,56.520,49.500,132.750,173.250,225.000,342.000
2024,"4,381.000",189.249,54.718,29.250,146.250,186.750,227.250,346.500
2025,"9,337.000",203.002,46.859,63.000,166.500,202.500,236.250,339.750
2026-R1,"6,933.000",158.410,43.636,38.250,126.000,153.000,182.250,339.750
2026-R2,"6,050.000",188.166,55.461,38.250,144.000,182.250,227.250,348.750


## 1.3 Institution information (Part A only)

The "programme" column is written in three different styles (` - `, `/`, brackets), so a rule-based parser splits it into type (public / private / abroad), province, institute, field and, for institutes abroad, the country. Repeated parts (`private/private`) are also fixed.

In [5]:
pa = clean[clean.exam.isin(C.PART_A_EXAMS)]
display(pd.crosstab(pa.exam, pa.inst_type, margins=True))
display(pd.crosstab(clean.exam, clean.field, margins=True))
pa[pa.inst_type == 'private'].groupby(['province', 'institute']).size().rename('rows').reset_index().head(10)

inst_type,abroad,private,public,unknown,All
exam,,,,,
2025,92,8097,1148,0,9337
2026-R1,50,6137,746,0,6933
2026-R2,85,4990,973,2,6050
All,227,19224,2867,2,22320


field,medicine,stomatology,unknown,All
exam,,,,
2023,4895,1987,0,6882
2024,3393,988,0,4381
2025,7005,2332,0,9337
2026-R1,4738,2195,0,6933
2026-R2,6048,0,2,6050
All,26079,7502,2,33583


,province,institute,rows
0,بغلان,حکیم سنایی,370
1,بلخ,اریا,507
2,بلخ,تاج,733
3,بلخ,رهنورد,845
4,بلخ,مولانا جلال الدین محمد بلخی,633
5,جوزجان,طلوع سعادت,20
6,خوست,احمد شاه ابدالی,525
7,غزنی,خاتم النبیین,258
8,غزنی,مسلم,455
9,ننګرهار,اریانا,1412


## 1.4 Record linkage

The candidate number changes between rounds (even between the two 2026 rounds), so candidates are linked by **name + father's name + grandfather's name** (plus field). Before linking, letter variants (Arabic/Persian/Pashto forms of yeh, kaf, gaf, hamza) and all spaces are normalised. If two people share all three names inside one round, they are not linked.

**How is the linkage validated?** A true link should:
1. keep the same **field** of study (random linking would match in about 65% of cases)
2. keep the same **institute** (Part A only)
3. have **correlated scores** (random links would show none)

In [6]:
print('Field agreement among matched name triples:', lk.field_agreement_by_key(clean))
print()
print('Linked records (score persistence, institute agreement):', lk.linkage_validity(h))

Field agreement among matched name triples: {'keys_matched': 7603, 'keys_with_conflicting_field': 8, 'observed_agreement': 0.9989477837695646, 'expected_if_random': 0.653014968442907}

Linked records (score persistence, institute agreement): {'linked_persons': 7600, 'pairs': 12454, 'spearman_linked': np.float64(0.5204228941635839), 'spearman_shuffled_null': np.float64(-0.065803540737091), 'institute_agreement': 0.9981449582615609, 'institute_pairs': 7547}


In [7]:
# Sensitivity: also link records whose grandfather name differs only by a typo (similarity >= 0.80)
cf, extra = lk.fuzzy_person_ids(clean)
print('extra links found by the typo-tolerant rule:', extra)
print('distinct persons - exact rule:', clean.person_id.nunique(), '| typo-tolerant rule:', cf.person_id_fuzzy.nunique())

extra links found by the typo-tolerant rule: 209
distinct persons - exact rule: 21129 | typo-tolerant rule: 20920


In [8]:
path = pipeline.export_clean(h)
print('anonymised file written to', path)
h[['exam', 'person_id', 'field', 'score', 'passed', 'attempt', 'prior_fails', 'returning']].head()

anonymised file written to d:\exit_exam_project\outputs\clean\clean_rounds_anonymized.csv


,exam,person_id,field,score,passed,attempt,prior_fails,returning
0,2023,00129858ee1f,stomatology,108.000,False,1,0,False
1,2023,0013b06a959a,medicine,256.500,True,1,0,False
2,2023,001423921f0d,medicine,218.250,True,1,0,False
3,2023,001be2cba303,medicine,193.500,False,1,0,False
4,2023,002327e93438,medicine,184.500,False,1,0,False


## 1.5 Summary

- Clean data: the per-round counts are in the `audit` table. Very few rows are removed.
- The pass mark of 216 is the same in all five rounds, so results can be compared across rounds in terms of pass/fail.
- The linkage is very reliable: field and institute agree for about 99.8-99.9% of linked pairs and scores are correlated. Allowing small typos adds only about 200 further links.
- **Limitation:** two different people with identical names could be treated as one person. The field and institute checks suggest this is rare, but it should be stated as a limitation in the paper.

Next: `02_part_A_three_rounds.ipynb` and `03_part_B_five_rounds.ipynb`.